# 长时序预测第一版代码：LSTM / Transformer / PatchTST-lite

本 notebook 结合课题报告要求，搭建一个可复现实验骨架：滑动窗口、时间顺序切分、训练集标准化、PyTorch 模型、MSE/MAE/MAPE 指标、预测曲线与残差分析。

第一版重点是把实验流程跑通，并为后续 Informer、Autoformer 和完整 PatchTST 扩展保留统一接口。

## 1. 导入依赖与全局配置

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def seed_everything(seed: int = 42):
    """固定随机种子，保证同一配置下的结果尽量可复现。"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


@dataclass
class ExperimentConfig:
    # 数据配置：没有真实 CSV 时，USE_SYNTHETIC_DEMO=True 会生成一份合成季节性数据。
    data_path: str | None = None
    time_col: str | None = None
    target_col: str = "target"
    use_synthetic_demo: bool = True

    # 预测任务配置：用 seq_len 个历史点预测 pred_len 个未来点。
    seq_len: int = 96
    pred_len: int = 24
    train_ratio: float = 0.7
    val_ratio: float = 0.1

    # 模型配置：model_name 可选 lstm、transformer、patchtst。
    model_name: str = "patchtst"
    d_model: int = 64
    hidden_size: int = 64
    num_layers: int = 2
    n_heads: int = 4
    dropout: float = 0.1
    patch_len: int = 16
    stride: int = 8
    use_decomposition: bool = True

    # 训练配置：第一版默认较小，方便在本机快速验证。
    batch_size: int = 32
    epochs: int = 5
    lr: float = 1e-3
    weight_decay: float = 1e-4
    seed: int = 42


cfg = ExperimentConfig()
seed_everything(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 2. 数据读取、标准化与滑动窗口

关键原则：验证集和测试集不能参与标准化参数拟合，否则会把未来分布信息泄露给模型。

In [ ]:
def make_synthetic_series(n: int = 2500) -> pd.DataFrame:
    """生成一份含趋势、日周期、周周期和外生变量的合成数据，用于无数据时验证代码。"""
    t = np.arange(n, dtype=np.float32)
    daily = np.sin(2 * np.pi * t / 24)
    weekly = np.sin(2 * np.pi * t / (24 * 7))
    trend = 0.0008 * t
    noise = 0.15 * np.random.randn(n).astype(np.float32)
    exog_temp = 0.6 * daily + 0.2 * np.random.randn(n).astype(np.float32)
    exog_calendar = np.cos(2 * np.pi * t / 24)
    target = 1.2 * daily + 0.8 * weekly + trend + 0.4 * exog_temp + noise
    return pd.DataFrame({
        "time": pd.date_range("2024-01-01", periods=n, freq="h"),
        "target": target,
        "exog_temp": exog_temp,
        "exog_calendar": exog_calendar,
    })


def load_dataframe(cfg: ExperimentConfig) -> pd.DataFrame:
    """读取 CSV 或生成合成数据；真实实验时把 data_path 指向 data/ 下的 CSV。"""
    if cfg.use_synthetic_demo or cfg.data_path is None:
        df = make_synthetic_series()
    else:
        df = pd.read_csv(cfg.data_path)

    if cfg.time_col and cfg.time_col in df.columns:
        df[cfg.time_col] = pd.to_datetime(df[cfg.time_col])
        df = df.sort_values(cfg.time_col).reset_index(drop=True)

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if cfg.target_col not in numeric_cols:
        raise ValueError(f"target_col={cfg.target_col!r} 必须是数值列，当前数值列为：{numeric_cols}")

    # 把目标列放在第 0 列，便于后面统一取 target_idx=0。
    feature_cols = [cfg.target_col] + [c for c in numeric_cols if c != cfg.target_col]
    return df[feature_cols].astype(np.float32)


class StandardScaler:
    """只保存训练集均值和标准差，避免引入 sklearn 依赖。"""
    def fit(self, values: np.ndarray):
        self.mean = values.mean(axis=0, keepdims=True)
        self.std = values.std(axis=0, keepdims=True)
        self.std[self.std < 1e-6] = 1.0
        return self

    def transform(self, values: np.ndarray) -> np.ndarray:
        return (values - self.mean) / self.std

    def inverse_target(self, values: np.ndarray, target_idx: int = 0) -> np.ndarray:
        return values * self.std[:, target_idx] + self.mean[:, target_idx]


class SlidingWindowDataset(Dataset):
    """把连续时间序列转成监督学习样本：X=[t-L,t)，y=[t,t+H)。"""
    def __init__(self, values: np.ndarray, seq_len: int, pred_len: int):
        self.values = torch.tensor(values, dtype=torch.float32)
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.max_start = len(values) - seq_len - pred_len + 1
        if self.max_start <= 0:
            raise ValueError("序列长度不足，无法构造滑动窗口，请调小 seq_len/pred_len 或增加数据量。")

    def __len__(self):
        return self.max_start

    def __getitem__(self, idx):
        x = self.values[idx: idx + self.seq_len]
        y = self.values[idx + self.seq_len: idx + self.seq_len + self.pred_len]
        return x, y


def build_dataloaders(cfg: ExperimentConfig):
    raw = load_dataframe(cfg).values
    n = len(raw)
    train_end = int(n * cfg.train_ratio)
    val_end = int(n * (cfg.train_ratio + cfg.val_ratio))

    scaler = StandardScaler().fit(raw[:train_end])
    values = scaler.transform(raw)

    # 验证/测试切片向前多保留 seq_len 个点，使第一个窗口有足够历史上下文。
    train_values = values[:train_end]
    val_values = values[max(0, train_end - cfg.seq_len):val_end]
    test_values = values[max(0, val_end - cfg.seq_len):]

    train_ds = SlidingWindowDataset(train_values, cfg.seq_len, cfg.pred_len)
    val_ds = SlidingWindowDataset(val_values, cfg.seq_len, cfg.pred_len)
    test_ds = SlidingWindowDataset(test_values, cfg.seq_len, cfg.pred_len)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)
    return train_loader, val_loader, test_loader, scaler, raw.shape[1]


train_loader, val_loader, test_loader, scaler, n_features = build_dataloaders(cfg)
n_features, len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset)

## 3. 指标函数

报告要求使用 MSE、MAE 和 MAPE。MAPE 对接近 0 的真实值敏感，因此分母加入很小的 epsilon。

In [ ]:
def regression_metrics(pred: torch.Tensor, true: torch.Tensor) -> dict[str, float]:
    """输入形状均为 [batch, pred_len, n_features]，这里只统计所有变量的平均误差。"""
    mse = torch.mean((pred - true) ** 2).item()
    mae = torch.mean(torch.abs(pred - true)).item()
    mape = torch.mean(torch.abs((pred - true) / torch.clamp(torch.abs(true), min=1e-5))).item() * 100
    return {"mse": mse, "mae": mae, "mape": mape}


def count_parameters(model: nn.Module) -> int:
    """统计可训练参数量，作为复杂度分析的第一项。"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 4. 模型一：LSTM 基线

LSTM 用于代表 RNN 架构。它按时间步递归建模，长预测窗口下容易出现长期依赖衰减，但参数量和实现复杂度较低。

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, n_features: int, hidden_size: int, num_layers: int, pred_len: int, dropout: float):
        super().__init__()
        self.pred_len = pred_len
        self.n_features = n_features
        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        # 用最后一个隐藏状态一次性回归未来 pred_len * n_features 个值。
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, pred_len * n_features),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, (h_n, _) = self.encoder(x)
        last_hidden = h_n[-1]
        out = self.head(last_hidden)
        return out.view(x.size(0), self.pred_len, self.n_features)

## 5. 模型二：Transformer 基线

Transformer 使用自注意力直接建模任意时间步之间的关系，但标准注意力复杂度约为 $O(L^2)$，长输入窗口下计算成本较高。

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class TransformerForecaster(nn.Module):
    def __init__(self, n_features: int, d_model: int, n_heads: int, num_layers: int, pred_len: int, dropout: float):
        super().__init__()
        self.pred_len = pred_len
        self.n_features = n_features
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_encoding = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, pred_len * n_features),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.input_proj(x)
        z = self.pos_encoding(z)
        z = self.encoder(z)
        pooled = z[:, -1]
        out = self.head(pooled)
        return out.view(x.size(0), self.pred_len, self.n_features)

## 6. 模型三：PatchTST-lite 与分解消融

PatchTST 的核心思想是把长序列切成 patch，降低 token 数并强化局部模式。这里实现教学版 PatchTST-lite：每个变量单独切 patch，经过共享 Transformer 编码后再预测未来。

In [ ]:
class MovingAverageDecomposition(nn.Module):
    """简单移动平均分解：trend 是平滑项，seasonal 是原序列减 trend。"""
    def __init__(self, kernel_size: int = 25):
        super().__init__()
        self.kernel_size = kernel_size
        self.padding = kernel_size // 2
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=self.padding, count_include_pad=False)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # AvgPool1d 期望 [B, C, L]，而我们的输入是 [B, L, C]。
        trend = self.avg(x.transpose(1, 2)).transpose(1, 2)
        if trend.size(1) != x.size(1):
            trend = trend[:, : x.size(1)]
        seasonal = x - trend
        return seasonal, trend


class PatchTSTLite(nn.Module):
    def __init__(
        self,
        n_features: int,
        seq_len: int,
        pred_len: int,
        patch_len: int,
        stride: int,
        d_model: int,
        n_heads: int,
        num_layers: int,
        dropout: float,
        use_decomposition: bool,
    ):
        super().__init__()
        self.n_features = n_features
        self.pred_len = pred_len
        self.patch_len = patch_len
        self.stride = stride
        self.use_decomposition = use_decomposition
        self.decomp = MovingAverageDecomposition(kernel_size=25)

        # unfold 后 patch 数由输入长度、patch_len 和 stride 决定。
        self.n_patches = 1 + max(0, (seq_len - patch_len) // stride)
        self.patch_proj = nn.Linear(patch_len, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches, d_model))

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.LayerNorm(self.n_patches * d_model),
            nn.Linear(self.n_patches * d_model, pred_len),
        )

    def _encode_one_stream(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, C] -> [B, C, L]，每个变量独立切 patch。
        x = x.transpose(1, 2)
        patches = x.unfold(dimension=-1, size=self.patch_len, step=self.stride)
        patches = patches[:, :, : self.n_patches, :]
        bsz, n_features, n_patches, patch_len = patches.shape

        # 合并 batch 和变量维度，共享同一个 patch encoder。
        patches = patches.reshape(bsz * n_features, n_patches, patch_len)
        z = self.patch_proj(patches) + self.pos_embed[:, :n_patches]
        z = self.encoder(z)
        out = self.head(z)
        return out.view(bsz, n_features, self.pred_len).transpose(1, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.use_decomposition:
            return self._encode_one_stream(x)

        # 分解消融：打开时分别预测 seasonal 和 trend，再相加得到最终预测。
        seasonal, trend = self.decomp(x)
        return self._encode_one_stream(seasonal) + self._encode_one_stream(trend)

## 7. 模型工厂、训练与评估

In [ ]:
def build_model(cfg: ExperimentConfig, n_features: int) -> nn.Module:
    if cfg.model_name == "lstm":
        return LSTMForecaster(n_features, cfg.hidden_size, cfg.num_layers, cfg.pred_len, cfg.dropout)
    if cfg.model_name == "transformer":
        return TransformerForecaster(n_features, cfg.d_model, cfg.n_heads, cfg.num_layers, cfg.pred_len, cfg.dropout)
    if cfg.model_name == "patchtst":
        return PatchTSTLite(
            n_features=n_features,
            seq_len=cfg.seq_len,
            pred_len=cfg.pred_len,
            patch_len=cfg.patch_len,
            stride=cfg.stride,
            d_model=cfg.d_model,
            n_heads=cfg.n_heads,
            num_layers=cfg.num_layers,
            dropout=cfg.dropout,
            use_decomposition=cfg.use_decomposition,
        )
    raise ValueError(f"未知模型：{cfg.model_name}")


def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    losses = []
    preds, trues = [], []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        pred = model(x)
        losses.append(loss_fn(pred, y).item())
        preds.append(pred.cpu())
        trues.append(y.cpu())
    preds = torch.cat(preds, dim=0)
    trues = torch.cat(trues, dim=0)
    metrics = regression_metrics(preds, trues)
    metrics["loss"] = float(np.mean(losses))
    return metrics, preds, trues


model = build_model(cfg, n_features).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
loss_fn = nn.MSELoss()

print(model.__class__.__name__)
print(f"trainable parameters: {count_parameters(model):,}")

In [ ]:
best_state = None
best_val = float("inf")
history = []
start_time = time.time()

for epoch in range(1, cfg.epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
    val_metrics, _, _ = evaluate(model, val_loader, loss_fn)
    row = {"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)

    if val_metrics["loss"] < best_val:
        best_val = val_metrics["loss"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(f"epoch={epoch:02d} train_loss={train_loss:.4f} val_mse={val_metrics['mse']:.4f} val_mae={val_metrics['mae']:.4f}")

elapsed = time.time() - start_time
if best_state is not None:
    model.load_state_dict(best_state)

pd.DataFrame(history), elapsed

## 8. 测试集评估、预测曲线与残差分析

In [ ]:
test_metrics, test_preds, test_trues = evaluate(model, test_loader, loss_fn)
test_metrics

In [ ]:
def plot_forecast_and_residual(preds: torch.Tensor, trues: torch.Tensor, scaler: StandardScaler, sample_idx: int = 0, target_idx: int = 0):
    """绘制单个窗口的目标变量预测曲线和残差曲线。"""
    pred = preds[sample_idx, :, target_idx].numpy()
    true = trues[sample_idx, :, target_idx].numpy()
    pred = scaler.inverse_target(pred, target_idx=target_idx)
    true = scaler.inverse_target(true, target_idx=target_idx)
    residual = true - pred

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    axes[0].plot(true, label="true", linewidth=2)
    axes[0].plot(pred, label="pred", linewidth=2)
    axes[0].set_title("Forecast vs. Ground Truth")
    axes[0].legend()

    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].plot(residual, label="residual", color="tab:red")
    axes[1].set_title("Residual")
    axes[1].legend()
    axes[1].set_xlabel("future step")
    plt.tight_layout()
    return fig


fig = plot_forecast_and_residual(test_preds, test_trues, scaler, sample_idx=0, target_idx=0)

## 9. 消融实验入口

将 `cfg.use_decomposition` 设为 `False` 后重新运行训练，即可得到 PatchTST-lite 不使用分解模块的结果。建议把两次 `test_metrics` 记录到表格中，用于报告中的关键组件有效性分析。

In [ ]:
summary = {
    "model": cfg.model_name,
    "seq_len": cfg.seq_len,
    "pred_len": cfg.pred_len,
    "use_decomposition": cfg.use_decomposition,
    "params": count_parameters(model),
    "train_seconds": round(elapsed, 2),
    **test_metrics,
}
pd.DataFrame([summary])

## 10. 后续扩展建议

- 增加 Informer：重点实现 ProbSparse Attention，记录长序列输入下的速度和显存变化。
- 增加 Autoformer：重点实现序列分解和 Auto-Correlation block，与当前 `MovingAverageDecomposition` 消融结果衔接。
- 增加多预测步长实验循环：例如 24、48、96、168、336，并自动保存每组指标。
- 增加单变量/多变量开关：只保留目标列即可做 univariate，对全部数值列建模即可做 multivariate。